In [ ]:
import base64
import hashlib
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys

import pandas as pd
import numpy as np
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Lipinski, rdMolDescriptors, AllChem, MACCSkeys
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

In [2]:

BASE_SEED = 20260915
DIELECTRIC_CONSTANT = 1
OPEN_CHEM_LIB_JAR = "../tools/datawarrior_all.jar"
OUTPUT_SDF = Path(f'../datasets/preprocessed_inxight/druglike_inxight_seeded_3d.sdf')

In [3]:
target_smiles = Chem.MolToSmiles(Chem.MolFromSmiles("CNC(=O)Oc1cccc2ccccc12"))
removed_total = 0

def is_organic_molecule(smiles):
    mol = Chem.MolFromSmiles(smiles)
    allowed = {1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 34, 35, 53}
    return mol is not None and mol.GetNumHeavyAtoms() >= 2 and len(Chem.GetMolFrags(mol)) == 1 and any(a.GetAtomicNum() == 6 for a in mol.GetAtoms()) and all(a.GetAtomicNum() in allowed for a in mol.GetAtoms())


for penetrance in ["penetrant", "nonpenetrant"]:
    df = pd.read_csv(f"../datasets/preprocessed_inxight/inxight_cns_{penetrance}.csv")

    canonical_smiles = df["SMILES"].map(lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)

    structure_mask = df["SMILES"].map(is_organic_molecule)
    print(f"{penetrance}: removing {(~structure_mask).sum()} non-organic or multicomponent structures")
    df = df.loc[structure_mask].copy()

    remove_mask = canonical_smiles.eq(target_smiles)
    print(f"{penetrance}: removing {remove_mask.sum()} matching molecule(s)")
    display(df.loc[remove_mask, [c for c in ["name", "ID", "SMILES"] if c in df.columns]])
    removed_total += remove_mask.sum()
    df = df.loc[~remove_mask].copy()

    mols = df["SMILES"].map(Chem.MolFromSmiles)
    df[["MW", "nHBAcc", "nHBDon", "SLogP", "TopoPSA"]] = [[Descriptors.ExactMolWt(m), Lipinski.NumHAcceptors(m), Lipinski.NumHDonors(m), Crippen.MolLogP(m), rdMolDescriptors.CalcTPSA(m)] for m in mols]
    df[["MW", "SLogP", "TopoPSA"]] = df[["MW", "SLogP", "TopoPSA"]].round(2)
    df[["nHBAcc", "nHBDon"]] = df[["nHBAcc", "nHBDon"]].astype(int)

    output_file_druglike_inxight = f"../datasets/preprocessed_inxight/druglike_inxight_{penetrance}.csv"
    base = df
    drug_like_mask = base["MW"].le(800) & base["nHBDon"].le(6) & base["nHBAcc"].le(11) & base["SLogP"].le(5) & base["SLogP"].gt(0) & base["TopoPSA"].lt(180)
    all_valid_dataset_drug_like = base.loc[drug_like_mask].copy()
    all_valid_dataset_not_drug_like = base.loc[~drug_like_mask].copy()
    all_valid_dataset_drug_like.to_csv(output_file_druglike_inxight, index=False)

    print("Cleaned valid dataset:", len(base))
    print("Drug-like subset:", len(all_valid_dataset_drug_like))
    print("Exact complement:", len(all_valid_dataset_not_drug_like))

assert removed_total == 1, f"Expected to remove exactly one molecule, but removed {removed_total}"

penetrant: removing 61 non-organic or multicomponent structures
penetrant: removing 1 matching molecule(s)


[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors
[17:41:18] WARNING: not removing hydrogen atom without neighbors


,name,ID,SMILES
888,CARBARIL,R890C8J3N1,CNC(=O)OC1=C2C=CC=CC2=CC=C1


Cleaned valid dataset: 950
Drug-like subset: 636
Exact complement: 314
nonpenetrant: removing 35 non-organic or multicomponent structures
nonpenetrant: removing 0 matching molecule(s)


,name,ID,SMILES


Cleaned valid dataset: 376
Drug-like subset: 228
Exact complement: 148


In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Configuration
PEN_CSV = Path("../datasets/preprocessed_inxight/druglike_inxight_penetrant.csv")
NONPEN_CSV = Path("../datasets/preprocessed_inxight/druglike_inxight_nonpenetrant.csv")
REFERENCE_CSV = Path("../datasets/druglike_b3db_labelled_outerfolds.csv")
OUTPUT_DIR = Path("../datasets/preprocessed_inxight/")
SIMILARITY_THRESHOLD, PAIRWISE_THRESHOLD, PAIRWISE_RESTARTS, SEED = 0.60, None, 200, 15  # Set PAIRWISE_THRESHOLD=0.60 if desired
NAME_COL, ID_COL, SMILES_COL = "name", "ID", "SMILES"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)

def parse_smiles(value):
    if pd.isna(value) or not str(value).strip(): return None, None
    mol = Chem.MolFromSmiles(str(value).strip())
    return (None, None) if mol is None else (mol, Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True))

def strip_fp(row):
    return {key: value for key, value in row.items() if key != "_fp"}

def load_external(path, class_value, class_name):
    df = pd.read_csv(path, low_memory=False)
    missing = {NAME_COL, ID_COL, SMILES_COL} - set(df.columns)
    if missing: raise KeyError(f"{path} is missing columns: {sorted(missing)}")
    valid, excluded, seen = [], [], {}
    for source_index, row in df.iterrows():
        mol, canonical = parse_smiles(row[SMILES_COL])
        base = {"name": row[NAME_COL], "ID": row[ID_COL], "SMILES": row[SMILES_COL], "bbb_class": class_value, "bbb_label": class_name, "source_file": path.name, "source_row_index": int(source_index)}
        if mol is None:
            excluded.append({**base, "canonical_SMILES": np.nan, "exclusion_reason": "invalid_external_smiles"})
        elif canonical in seen:
            excluded.append({**base, "canonical_SMILES": canonical, "duplicate_of_source_row": seen[canonical], "exclusion_reason": "duplicate_within_class"})
        else:
            seen[canonical] = int(source_index)
            valid.append({**base, "canonical_SMILES": canonical, "_fp": fpgen.GetFingerprint(mol)})
    return valid, excluded

def filter_against_reference(rows, reference_fps, reference_smiles, reference_indices):
    kept, excluded = [], []
    for count, row in enumerate(rows, 1):
        similarities = DataStructs.BulkTanimotoSimilarity(row["_fp"], reference_fps)
        closest = int(np.argmax(similarities))
        annotated = {**row, "max_reference_tanimoto": float(similarities[closest]), "closest_reference_row_index": int(reference_indices[closest]), "closest_reference_canonical_SMILES": reference_smiles[closest]}
        if annotated["max_reference_tanimoto"] >= SIMILARITY_THRESHOLD:
            excluded.append({**strip_fp(annotated), "exclusion_reason": f"reference_similarity_ge_{SIMILARITY_THRESHOLD:g}"})
        else:
            kept.append(annotated)
        if count % 500 == 0: print(f"  Processed {count}/{len(rows)}")
    return kept, excluded

def random_balance(penetrant, nonpenetrant):
    rng, n = np.random.default_rng(SEED), min(len(penetrant), len(nonpenetrant))
    if n == 0: raise RuntimeError("At least one class is empty after filtering.")
    pen_indices = set(map(int, rng.choice(len(penetrant), n, replace=False)))
    nonpen_indices = set(map(int, rng.choice(len(nonpenetrant), n, replace=False)))
    selected_pen = [row for i, row in enumerate(penetrant) if i in pen_indices]
    selected_nonpen = [row for i, row in enumerate(nonpenetrant) if i in nonpen_indices]
    excluded = [{**strip_fp(row), "exclusion_reason": "not_selected_for_class_balance"} for i, row in enumerate(penetrant) if i not in pen_indices]
    excluded += [{**strip_fp(row), "exclusion_reason": "not_selected_for_class_balance"} for i, row in enumerate(nonpenetrant) if i not in nonpen_indices]
    return selected_pen, selected_nonpen, excluded

def pairwise_balance(penetrant, nonpenetrant):
    best_pen, best_nonpen = [], []
    for restart in range(PAIRWISE_RESTARTS):
        rng, selected, selected_fps = np.random.default_rng(SEED + restart), {0: [], 1: []}, []
        orders = {0: rng.permutation(len(penetrant)), 1: rng.permutation(len(nonpenetrant))}
        rows = {0: penetrant, 1: nonpenetrant}
        for class_value in [0, 1] * max(len(penetrant), len(nonpenetrant)):
            for index in orders[class_value]:
                candidate = rows[class_value][int(index)]
                if candidate in selected[class_value]: continue
                if selected_fps and max(DataStructs.BulkTanimotoSimilarity(candidate["_fp"], selected_fps)) >= PAIRWISE_THRESHOLD: continue
                selected[class_value].append(candidate); selected_fps.append(candidate["_fp"]); break
        n = min(len(selected[0]), len(selected[1]))
        if n > len(best_pen): best_pen, best_nonpen = selected[0][:n], selected[1][:n]
    selected_keys = {(row["bbb_class"], row["canonical_SMILES"]) for row in best_pen + best_nonpen}
    excluded = [{**strip_fp(row), "exclusion_reason": "not_selected_for_pairwise_external_dissimilarity"} for row in penetrant + nonpenetrant if (row["bbb_class"], row["canonical_SMILES"]) not in selected_keys]
    return best_pen, best_nonpen, excluded

# Load and fingerprint the B3DB reference set
reference_df = pd.read_csv(REFERENCE_CSV, low_memory=False)
reference_smiles_col = next((column for column in ["SMILES", "smiles", "canonical_smiles", "canonical_SMILES"] if column in reference_df.columns), None)
if reference_smiles_col is None: raise KeyError("No reference SMILES column found.")

reference_fps, reference_smiles, reference_indices, invalid_reference = [], [], [], []
seen_reference = set()
for index, raw_smiles in enumerate(reference_df[reference_smiles_col]):
    mol, canonical = parse_smiles(raw_smiles)
    if mol is None:
        invalid_reference.append({"source_row_index": index, "raw_SMILES": raw_smiles, "exclusion_reason": "invalid_reference_smiles"})
    elif canonical not in seen_reference:
        seen_reference.add(canonical); reference_fps.append(fpgen.GetFingerprint(mol)); reference_smiles.append(canonical); reference_indices.append(index)

print("Unique valid reference structures:", len(reference_fps))

# Load, deduplicate and remove structures with conflicting external labels
penetrant, excluded_pen = load_external(PEN_CSV, 0, "penetrant")
nonpenetrant, excluded_nonpen = load_external(NONPEN_CSV, 1, "nonpenetrant")
conflicts = {row["canonical_SMILES"] for row in penetrant} & {row["canonical_SMILES"] for row in nonpenetrant}
excluded_conflicts = [{**strip_fp(row), "exclusion_reason": "same_structure_has_conflicting_class_labels"} for row in penetrant + nonpenetrant if row["canonical_SMILES"] in conflicts]
penetrant = [row for row in penetrant if row["canonical_SMILES"] not in conflicts]
nonpenetrant = [row for row in nonpenetrant if row["canonical_SMILES"] not in conflicts]

# Remove compounds too similar to B3DB
penetrant, excluded_pen_similarity = filter_against_reference(penetrant, reference_fps, reference_smiles, reference_indices)
nonpenetrant, excluded_nonpen_similarity = filter_against_reference(nonpenetrant, reference_fps, reference_smiles, reference_indices)
print(f"After reference filtering: penetrant={len(penetrant)}, nonpenetrant={len(nonpenetrant)}")

# Select the balanced final set
if PAIRWISE_THRESHOLD is None:
    selected_pen, selected_nonpen, excluded_balance = random_balance(penetrant, nonpenetrant)
else:
    selected_pen, selected_nonpen, excluded_balance = pairwise_balance(penetrant, nonpenetrant)

final_rows = selected_pen + selected_nonpen
if any(row["max_reference_tanimoto"] >= SIMILARITY_THRESHOLD for row in final_rows): raise RuntimeError("Reference-similarity violation.")
if len({row["canonical_SMILES"] for row in final_rows}) != len(final_rows): raise RuntimeError("Duplicate structures remain.")

if PAIRWISE_THRESHOLD is not None:
    final_fps = [row["_fp"] for row in final_rows]
    for i in range(1, len(final_fps)):
        if max(DataStructs.BulkTanimotoSimilarity(final_fps[i], final_fps[:i])) >= PAIRWISE_THRESHOLD: raise RuntimeError("Pairwise-similarity violation.")

# Save outputs
final_df = pd.DataFrame([strip_fp(row) for row in final_rows]).sample(frac=1, random_state=SEED).reset_index(drop=True)
final_df.insert(0, "external_row_index", np.arange(len(final_df)))
penetrant_df = final_df[final_df["bbb_class"] == 0].reset_index(drop=True)
nonpenetrant_df = final_df[final_df["bbb_class"] == 1].reset_index(drop=True)
excluded_df = pd.DataFrame(excluded_pen + excluded_nonpen + excluded_conflicts + excluded_pen_similarity + excluded_nonpen_similarity + excluded_balance)

final_df.to_csv(OUTPUT_DIR / "inxight_druglike_balanced.csv", index=False)
# penetrant_df.to_csv(OUTPUT_DIR / "inxight_external_balanced_penetrant.csv", index=False)
# nonpenetrant_df.to_csv(OUTPUT_DIR / "inxight_external_balanced_nonpenetrant.csv", index=False)
# excluded_df.to_csv(OUTPUT_DIR / "inxight_external_excluded.csv", index=False)
if invalid_reference: pd.DataFrame(invalid_reference).to_csv(OUTPUT_DIR / "invalid_reference_smiles.csv", index=False)

summary = {
    "reference_csv": str(REFERENCE_CSV.resolve()),
    "penetrant_csv": str(PEN_CSV.resolve()),
    "nonpenetrant_csv": str(NONPEN_CSV.resolve()),
    "similarity_definition": {"fingerprint": "Morgan", "radius": 2, "bits": 2048, "include_chirality": True, "metric": "Tanimoto"},
    "reference_exclusion_rule": f"exclude if max Tanimoto >= {SIMILARITY_THRESHOLD}",
    "pairwise_external_threshold": PAIRWISE_THRESHOLD,
    "seed": SEED,
    "n_reference_unique_valid": len(reference_fps),
    "n_final_penetrant": len(penetrant_df),
    "n_final_nonpenetrant": len(nonpenetrant_df),
    "n_final_total": len(final_df),
    "max_reference_similarity_in_final_set": float(final_df["max_reference_tanimoto"].max()),
    "exclusion_counts": excluded_df["exclusion_reason"].value_counts().to_dict() if not excluded_df.empty else {},
}
(OUTPUT_DIR / "selection_summary.json").write_text(json.dumps(summary, indent=2))
excluded_df.to_csv(OUTPUT_DIR / "inxight_external_excluded.csv", index=False)

print(f"Final balanced set: {len(penetrant_df)} penetrant + {len(nonpenetrant_df)} nonpenetrant = {len(final_df)}")
print("Maximum reference similarity:", f"{summary['max_reference_similarity_in_final_set']:.4f}")
print("Saved to:", OUTPUT_DIR.resolve())

Unique valid reference structures: 3188
  Processed 500/636
After reference filtering: penetrant=328, nonpenetrant=105
Final balanced set: 105 penetrant + 105 nonpenetrant = 210
Maximum reference similarity: 0.5862
Saved to: /Users/sziadat/Documents/bbbx/bbb-exc-score/datasets/preprocessed_inxight


##### Generate OpenChemLib 3D conformers


In [5]:
# Use the jar from the SAME DataWarrior version used for the manuscript.
# Leave as None to search the macOS app and launcher-managed update folder.
# Typical macOS path: /Applications/DataWarrior.app/Contents/Java/update/datawarrior_all.jar


def find_openchemlib_jar(explicit_path=None):
    # DataWarrior may package OpenChemLib in openchemlib.jar OR in a larger
    # DataWarrior jar. Test jar contents instead of relying on the filename.
    required_classes = {
        'org/openmolecules/chem/conf/gen/ConformerGenerator.class',
        'com/actelion/research/chem/forcefield/mmff/ForceFieldMMFF94.class',
    }

    def contains_required_classes(path):
        try:
            with zipfile.ZipFile(path) as jar:
                names = set(jar.namelist())
            return required_classes.issubset(names)
        except (OSError, zipfile.BadZipFile):
            return False

    candidates = []
    if explicit_path:
        explicit_path = Path(explicit_path).expanduser()
        if not explicit_path.is_file():
            raise FileNotFoundError(f'OPEN_CHEM_LIB_JAR does not exist: {explicit_path}')
        if not contains_required_classes(explicit_path):
            raise ValueError(f'The selected jar does not contain the required OpenChemLib classes: {explicit_path}')
        return explicit_path.resolve()
    if os.environ.get('OPENCHEMLIB_JAR'):
        candidates.append(Path(os.environ['OPENCHEMLIB_JAR']).expanduser())


    # Spotlight fallback catches launcher-managed jars stored in an unexpected
    # macOS user directory without recursively scanning the entire home folder.
    mdfind = shutil.which('mdfind')
    if mdfind:
        query = "kMDItemFSName == '*openchemlib*.jar'cd || kMDItemFSName == '*datawarrior*.jar'cd"
        result = subprocess.run([mdfind, query], capture_output=True, text=True, check=False)
        candidates.extend(Path(line) for line in result.stdout.splitlines() if line.strip())

    # Prefer a jar explicitly named for OpenChemLib if more than one valid jar exists.
    candidates = sorted(set(candidates), key=lambda p: ('openchemlib' not in p.name.lower(), str(p)))
    for candidate in candidates:
        if candidate.is_file() and contains_required_classes(candidate):
            return candidate.resolve()
    raise FileNotFoundError(
        'No installed jar containing the required OpenChemLib conformer classes was found. '
        'The usual macOS target is /Applications/DataWarrior.app/Contents/Java/update/datawarrior_all.jar. '
        'In Terminal, run: find /Applications/DataWarrior.app ~/Library -type f -name \"*.jar\" 2>/dev/null '
        'Then set OPEN_CHEM_LIB_JAR to the matching DataWarrior/OpenChemLib jar path.'
    )

def conformer_seed(smiles, base_seed=BASE_SEED):
    digest = hashlib.sha256(f'{base_seed}\0{smiles}'.encode()).digest()[:8]
    seed = int.from_bytes(digest, 'big', signed=True)
    return seed if seed != 0 else 1

def b64(value):
    return base64.b64encode(str(value).encode('utf-8')).decode('ascii')

def find_java_runtime(ocl_jar):
    # Prefer the Java runtime bundled with the selected DataWarrior app.
    # /usr/bin/java on macOS may be only a placeholder that prints
    # 'Unable to locate a Java Runtime' and exits.
    ocl_jar = Path(ocl_jar).resolve()
    candidates = []
    app_bundle = next((parent for parent in ocl_jar.parents if parent.suffix == '.app'), None)
    if app_bundle is not None:
        contents = app_bundle / 'Contents'
        candidates.extend(
            path for path in contents.rglob('java')
            if path.is_file() and path.parent.name == 'bin'
        )

    system_java = shutil.which('java')
    if system_java:
        candidates.append(Path(system_java))

    diagnostics = []
    for candidate in dict.fromkeys(candidates):
        if not os.access(candidate, os.X_OK):
            continue
        version = subprocess.run(
            [str(candidate), '-version'], capture_output=True, text=True, check=False
        )
        if version.returncode != 0:
            diagnostics.append(f'{candidate}: {version.stderr.strip()}')
            continue
        modules = subprocess.run(
            [str(candidate), '--list-modules'], capture_output=True, text=True, check=False
        )
        if modules.returncode == 0 and 'jdk.compiler@' in modules.stdout:
            return candidate.resolve()
        diagnostics.append(f'{candidate}: Java found, but the jdk.compiler module is absent')

    detail = '\n'.join(diagnostics) if diagnostics else 'No Java executable was found.'
    raise RuntimeError(
        'A working Java runtime with the jdk.compiler module is required. '
        'The notebook checked DataWarrior’s bundled runtime before the system Java.\n' + detail
    )

def pack_sdf_properties(row):
    """Encode every original CSV column/value for the Java helper."""
    return "\u001e".join(
        f"{column}\u001f{row[column]}"
        for column in row.index
    )

In [6]:
JAVA_SOURCE = r'''
import com.actelion.research.chem.MolfileV3Creator;
import com.actelion.research.chem.StereoMolecule;
import com.actelion.research.chem.SmilesParser;
import com.actelion.research.chem.forcefield.mmff.ForceFieldMMFF94;
import org.openmolecules.chem.conf.gen.ConformerGenerator;
import org.openmolecules.chem.conf.gen.RigidFragmentCache;
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;

public class SeededDataWarriorConformers {
  static final int MAX_TORSION_SETS = 1000;
  static final int MAX_ITERATIONS = 10000;
  static final double GRADIENT_TOLERANCE = 0.0001;
  static final double FUNCTION_TOLERANCE = 1.0e-6;

  public static void main(String[] args) throws Exception {
    Path manifest = Path.of(args[0]);
    Path output = Path.of(args[1]);
    Path failures = Path.of(args[2]);
    double dielectric = Double.parseDouble(args[3]);
    RigidFragmentCache.getDefaultInstance().loadDefaultCache();
    ForceFieldMMFF94.initialize(ForceFieldMMFF94.MMFF94SPLUS);
    Map<String,Object> options = new HashMap<>();
    options.put("dielectric constant", dielectric);
    int inputCount = 0, writtenCount = 0, failureCount = 0;

    try (BufferedReader in = Files.newBufferedReader(manifest, StandardCharsets.UTF_8);
         BufferedWriter sdf = Files.newBufferedWriter(output, StandardCharsets.UTF_8);
         BufferedWriter failed = Files.newBufferedWriter(failures, StandardCharsets.UTF_8)) {
      failed.write("dataframe_row\tinput_id\tsmiles\terror\n");
      String line;
      while ((line = in.readLine()) != null) {
        inputCount++;
        String[] f = line.split("\\t", -1);
        String row = decode(f[0]), id = decode(f[1]), smiles = decode(f[2]);
        long seed = Long.parseLong(f[3]);
        String originalProperties = decode(f[4]);

        try {
          StereoMolecule mol = generateConformer(smiles, seed);
          double energy = Double.NaN;
          
          String minimizationError = "";

          try {
            energy = minimize(mol, options);
          } catch (Exception e) {
            // Match DataWarrior: retain the generated conformer when
            // MMFF94 typing or minimization fails.
            minimizationError = e.getClass().getSimpleName() + ": " + e.getMessage();
          }

          center(mol);
          mol.setName(id.isBlank() ? "row " + row : id);
          sdf.write(new MolfileV3Creator(mol).getMolfile());
          // Restore every original CSV column using its exact original name.
          for (String entry : originalProperties.split("\u001e", -1)) {
            if (entry.isEmpty()) continue;

            int separator = entry.indexOf('\u001f');
            if (separator < 0) continue;

            String columnName = entry.substring(0, separator);
            String columnValue = entry.substring(separator + 1);

            property(sdf, columnName, columnValue);
          }
          property(sdf, "DATAFRAME_ROW", row);
          property(sdf, "INPUT_ID", id);
          property(sdf, "INPUT_SMILES", smiles);
          property(sdf, "CONFORMER_SEED", Long.toString(seed));
          property(sdf, "CONFORMER_ALGORITHM", "systematic, low-energy biased");
          property(sdf, "TORSION_SOURCE", "crystallographic");
          property(sdf, "FORCE_FIELD", "MMFF94s+");
          property(sdf, "DIELECTRIC_CONSTANT", Double.toString(dielectric));
          property(
              sdf,
              "MMFF94_ENERGY",
              Double.isNaN(energy) ? "" : String.format(Locale.ROOT, "%.9g", energy)
          );
          property(sdf, "MINIMIZATION_ERROR", minimizationError);
          sdf.write("$$$$\n");
          writtenCount++;
        } catch (Throwable e) {
          failureCount++;
          failed.write(clean(row)+"\t"+clean(id)+"\t"+clean(smiles)+"\t"
              +clean(e.getClass().getSimpleName()+": "+e.getMessage())+"\n");
        }
      }
    }
    System.out.printf(Locale.ROOT, "Input: %d  Written: %d  Failed: %d%n",
        inputCount, writtenCount, failureCount);
    if (failureCount != 0) System.exit(1);
  }

  static StereoMolecule generateConformer(String smiles, long seed) throws Exception {
    int[] strategies = {
        ConformerGenerator.STRATEGY_LIKELY_SYSTEMATIC,
        ConformerGenerator.STRATEGY_LIKELY_RANDOM,
        ConformerGenerator.STRATEGY_ADAPTIVE_RANDOM
    };

    for (int i = 0; i < strategies.length; i++) {
      StereoMolecule candidate = new StereoMolecule();
      new SmilesParser().parse(candidate, smiles);

      ConformerGenerator cg = new ConformerGenerator(seed + i, true);

      if (!cg.initializeConformers(candidate, strategies[i], MAX_TORSION_SETS, false))
        continue;

      StereoMolecule conformer = cg.getNextConformerAsMolecule(candidate);

      if (conformer != null)
        return conformer;
    }

    throw new Exception("No conformer generated with any OpenChemLib strategy");
  }

  static double minimize(StereoMolecule mol, Map<String,Object> options) throws Exception {
    int[] fragmentNo = new int[mol.getAllAtoms()];
    int fragmentCount = mol.getFragmentNumbers(fragmentNo, false, true);
    double energy = Double.NaN;
    if (fragmentCount == 1) {
      ForceFieldMMFF94 ff = new ForceFieldMMFF94(mol, ForceFieldMMFF94.MMFF94SPLUS, options);
      int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);
      if (error != 0) throw new Exception("MMFF94 error code " + error);
      energy = ff.getTotalEnergy();
    } else {
      StereoMolecule[] fragments = mol.getFragments(fragmentNo, fragmentCount);
      int maxAtoms = 0;
      for (StereoMolecule fragment : fragments) {
        if (fragment.getAllAtoms() > 2) {
          ForceFieldMMFF94 ff = new ForceFieldMMFF94(fragment, ForceFieldMMFF94.MMFF94SPLUS, options);
          int error = ff.minimise(MAX_ITERATIONS, GRADIENT_TOLERANCE, FUNCTION_TOLERANCE);
          if (error != 0) throw new Exception("MMFF94 error code " + error);
          if (maxAtoms < fragment.getAllAtoms()) {
            maxAtoms = fragment.getAllAtoms(); energy = ff.getTotalEnergy();
          }
        }
      }
      int[] atomInFragment = new int[fragmentCount];
      for (int atom=0; atom<fragmentNo.length; atom++) {
        int fragment = fragmentNo[atom], sourceAtom = atomInFragment[fragment]++;
        mol.setAtomX(atom, fragments[fragment].getAtomX(sourceAtom));
        mol.setAtomY(atom, fragments[fragment].getAtomY(sourceAtom));
        mol.setAtomZ(atom, fragments[fragment].getAtomZ(sourceAtom));
      }
    }
    return energy;
  }

  static void center(StereoMolecule mol) {
    double x=0, y=0, z=0;
    for (int a=0; a<mol.getAllAtoms(); a++) { x+=mol.getAtomX(a); y+=mol.getAtomY(a); z+=mol.getAtomZ(a); }
    x/=mol.getAllAtoms(); y/=mol.getAllAtoms(); z/=mol.getAllAtoms();
    for (int a=0; a<mol.getAllAtoms(); a++) {
      mol.setAtomX(a,mol.getAtomX(a)-x); mol.setAtomY(a,mol.getAtomY(a)-y); mol.setAtomZ(a,mol.getAtomZ(a)-z);
    }
  }

  static void property(BufferedWriter w, String name, String value) throws Exception {
    w.write(">  <"+name+">\n"+clean(value)+"\n\n");
  }
  static String decode(String s) { return new String(Base64.getDecoder().decode(s), StandardCharsets.UTF_8); }
  static String clean(String s) { return s == null ? "" : s.replace('\t',' ').replace('\r',' ').replace('\n',' '); }
}
'''

print('Seeded OpenChemLib helper defined.')

Seeded OpenChemLib helper defined.


In [7]:
# Generate one reproducible 3D conformer for every row in the drug-like dataset.
druglike_balanced_inxight = "../datasets/preprocessed_inxight/inxight_druglike_balanced.csv"
ocl_jar = find_openchemlib_jar(OPEN_CHEM_LIB_JAR)

java_candidates = [
    path for path in Path(sys.prefix).rglob("java")
    if path.is_file() and path.parent.name == "bin"
]

if not java_candidates:
    raise FileNotFoundError(
        f"Java was not installed under the active environment: {sys.prefix}"
    )

java = java_candidates[0]
print("Using Java:", java)


OUTPUT_SDF = OUTPUT_SDF.resolve()
OUTPUT_SDF.parent.mkdir(parents=True, exist_ok=True)
failures_file = OUTPUT_SDF.with_suffix(OUTPUT_SDF.suffix + '.failures.tsv')

# NO. is the stable B3DB identifier; the DataFrame index is also retained explicitly.
# Read as strings so every value and column name is preserved as written
sdf_source_df = pd.read_csv(
    druglike_balanced_inxight,
    dtype=str,
    keep_default_na=False,
)

id_column = "NO." if "NO." in sdf_source_df.columns else None

with tempfile.TemporaryDirectory(prefix="seeded_openchemlib_") as tmp:
    tmp = Path(tmp)
    java_file = tmp / "SeededDataWarriorConformers.java"
    manifest = tmp / "manifest.tsv"
    java_file.write_text(JAVA_SOURCE, encoding="utf-8")

    with manifest.open("w", encoding="utf-8", newline="") as handle:
        for dataframe_row, row in sdf_source_df.iterrows():
            smiles = row["SMILES"].strip()
            input_id = row[id_column] if id_column else dataframe_row
            seed = conformer_seed(smiles)
            properties = pack_sdf_properties(row)

            handle.write(
                f"{b64(dataframe_row)}\t"
                f"{b64(input_id)}\t"
                f"{b64(smiles)}\t"
                f"{seed}\t"
                f"{b64(properties)}\n"
            )

    command = [
        java, '--class-path', str(ocl_jar), str(java_file),
        str(manifest), str(OUTPUT_SDF), str(failures_file), str(DIELECTRIC_CONSTANT),
    ]
    print('Using OpenChemLib:', ocl_jar)
    print('Using Java:', java)
    print('Base seed:', BASE_SEED)
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.stdout.strip():
        print(result.stdout.strip())

if result.returncode != 0:
    error_detail = result.stderr.strip() or 'Java exited without an error message.'
    if failures_file.exists() and failures_file.stat().st_size > 0:
        error_detail += f'\nPer-molecule failures, if any: {failures_file}'
    raise RuntimeError(f'Conformer generation failed:\n{error_detail}')

# Verify the critical one-input-row/one-SDF-record relationship.
with OUTPUT_SDF.open('r', encoding='utf-8') as handle:
    sdf_record_count = sum(line.rstrip() == '$$$$' for line in handle)
assert sdf_record_count == len(final_df), (
    f'SDF has {sdf_record_count} records but the input has {len(all_valid_dataset_drug_like)} rows.'
)
if failures_file.exists() and failures_file.read_text(encoding='utf-8').count('\n') <= 1:
    failures_file.unlink()
print(f'Wrote {sdf_record_count} reproducible V3000 records to {OUTPUT_SDF}')

Using Java: /Users/sziadat/miniforge3/envs/bbbx_env_revisions/lib/jvm/bin/java
Using OpenChemLib: /Users/sziadat/Documents/bbbx/bbb-exc-score/tools/datawarrior_all.jar
Using Java: /Users/sziadat/miniforge3/envs/bbbx_env_revisions/lib/jvm/bin/java
Base seed: 20260915
Input: 210  Written: 210  Failed: 0
Wrote 210 reproducible V3000 records to /Users/sziadat/Documents/bbbx/bbb-exc-score/datasets/preprocessed_inxight/druglike_inxight_seeded_3d.sdf


In [8]:
from rdkit.Chem.PandasTools import LoadSDF
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.warning')   
data_warrior_3D_file = OUTPUT_SDF
# Load the unlabelled dataset
df = LoadSDF(data_warrior_3D_file, smilesName='SMILES')
df["SMILES"] = df["INPUT_SMILES"]
SMILES = df["SMILES"].copy()

supervised_label_df = df[["external_row_index", "SMILES", "bbb_class"]].copy()
#supervised_label_df = df[['SMILES', 'Class', 'logBB']] #'logBB', 


In [9]:
RDLogger.DisableLog('rdApp.warning')   


# ECFP
fpgen = AllChem.GetMorganGenerator(radius=2)
ecfp_fingerprints = []
for smile in SMILES:
    mol = Chem.MolFromSmiles(smile)
    fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2).ToBitString()
    ecfp_fingerprints.append(list(fp))

ecfp_col_names = [f"ECFP_{i}" for i in range(0,2048)]
ecfp_df = pd.DataFrame(ecfp_fingerprints, columns=ecfp_col_names)

# Mordred
mordred_output_file = "../datasets/preprocessed_inxight/inxight_drug_like_3d_mordred_labels.csv"

! python -m mordred -3 {data_warrior_3D_file} -o {mordred_output_file}
mordred_df = pd.read_csv(mordred_output_file)


# MACCS
def get_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol).ToList())

col_names = [f"MACCS_{i}" for i in range(0,167)]
maccs_fps = []
for smile in SMILES: 
    maccs_fps.append(get_maccs_fingerprint(smile))

maccs_fp_df = pd.DataFrame(data=maccs_fps, columns=col_names)


assert len(mordred_df) == len(df)
assert (mordred_df["name"].astype(str).reset_index(drop=True) == df["INPUT_ID"].astype(str).reset_index(drop=True)).all()

# Save dataset:
all_labelled_df = pd.concat([supervised_label_df, ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
druglike_labelled_ds = "../datasets/inxight_labelled_balanced.csv"
all_labelled_df.to_csv(druglike_labelled_ds,index=False,)


[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[17:42:20] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol